In [1]:
import numpy as np
import time
from gridcp.detector import GridDetector, DetectorState
from gridcp.scores import CUSUM
from gridcp.typing import ArrayLike
import gridcp

In [2]:
def run_online_grid_detector(
    data: ArrayLike,
    detector: GridDetector,
    reset_on_alarm: bool = False,
) -> tuple[DetectorState, dict]:
    """Run a configured GridDetector over a dataset sequentially.

    Parameters
    ----------
    data : ArrayLike
        Sequence of observations. Each element is passed as `x` to
        `detector.update`. For univariate data this is a 1D array of scalars;
        for multivariate data it should be a 2D array of shape (n_samples, n_features).
    detector : GridDetector
        A fully configured detector instance (with `score` and `threshold`
        set). State is initialized internally via `detector.init_state()`.
    reset_on_alarm : bool, optional
        If True, the detector state is reset to a fresh initial state immediately
        after an alarm is raised. This allows the detector to restart tracking
        from the next observation after each detected changepoint.
        Default is False.

    Returns
    -------
    state : DetectorState
        State after processing the full input sequence.
    output : dict
        Output dictionary from the final observation. It contains:
          - ``"index"``: time index (n_samples) after update.
          - ``"alarm"``: bool, True when ``max_score > threshold``.
          - ``"max_score"``: highest penalized score among active candidates.
          - ``"max_score_index"``: grid position of the highest-scoring candidate.
    """
    data = np.asarray(data)
    state = detector.init_state()
    output = None
    for x in data:
        state, output = detector.update(state, x)

    return state, output

In [3]:
def demo(n_samples=100, n_features=1, n_runs=100, reset_on_alarm=False):
    """Simulate toy univariate data with a known mean shift and run the grid detector."""
    rng = np.random.default_rng(seed=42)
    final_max_scores = []
    for _ in range(n_runs):
        n_pre, n_post = n_samples // 2, n_samples // 2 + n_samples % 2
        data = np.concatenate(
            [
                rng.normal(loc=0.0, scale=1.0, size=(n_pre, n_features)),
                rng.normal(loc=0.0, scale=1.0, size=(n_post, n_features)),
            ]
        )

        score = CUSUM(n_features)
        detector = GridDetector(score=score, threshold=1000.0)
        state, output = run_online_grid_detector(data, detector, reset_on_alarm)
        final_max_scores.append(output["max_score"])

    return final_max_scores

In [4]:
n_samples = 1000
n_runs = 500

In [5]:
start = time.perf_counter()
demo(n_samples=n_samples, n_runs=n_runs, reset_on_alarm=False)
stop = time.perf_counter()
print(f"Execution time: {stop - start:.4f} seconds")

Execution time: 6.0025 seconds


In [ ]:
import gridcp.old_api as old_api

detector = old_api.make_univariate_mean_change_detector()
n_paths_calibration = n_runs
start = time.perf_counter()
detector.calibrate_false_alarm(
    false_alarm_probability=0.05,
    N=n_samples,
    K=n_paths_calibration,
    null_dist=np.random.normal,
)
stop = time.perf_counter()
print(f"Execution time: {stop - start:.4f} seconds")

Execution time: 2.2255 seconds


In [ ]:
np.random.seed(42)
n_samples = 500
n_pre = n_samples // 2
data = np.concatenate(
    [
        np.random.normal(loc=0.0, scale=1.0, size=n_pre),
        np.random.normal(loc=0.0, scale=1.0, size=n_samples - n_pre),
    ]
)

old_detector = gridcp.old_api.make_univariate_mean_change_detector(
    penalty_constant=10.0
)
new_score = CUSUM(n_features=1)
new_detector = GridDetector(score=new_score, threshold=10.0)
state = new_detector.init_state()
output = None

for x in data:
    old_detector.update(x)
    state, output = new_detector.update(state, x)
    assert state.grid == [-g - 1 for g in old_detector._state["grid_list"]], (
        "Grid lists do not match between old and new detectors."
    )

    assert np.isclose(state.current_score_state.sum, old_detector._state["sum"]), (
        "Score sums do not match between old and new detectors."
    )

    for x, y in zip(old_detector._state["sum_pre_list"], state.previous_score_states):
        assert np.isclose(x, y.sum), (
            "Sum pre lists do not match between old and new detectors."
        )

In [ ]:
# Unknown-variance univariate mean-change: old API vs new API equivalence check
import numpy as np
import gridcp
from gridcp.detector import GridDetector as NewGridDetector
from gridcp.scores import GaussianMean

np.random.seed(42)
n_samples = 500
n_pre = n_samples // 2
data = np.concatenate(
    [
        np.random.normal(loc=0.0, scale=1.0, size=n_pre),
        np.random.normal(loc=0.0, scale=1.0, size=n_samples - n_pre),
    ]
)

old_detector = gridcp.old_api.make_univariate_mean_change_detector(
    penalty_constant=10.0, mode="unknown_variance"
)
new_score = GaussianMean()
new_detector = NewGridDetector(score=new_score, threshold=10.0)
state = new_detector.init_state()
output = None

for x in data:
    old_detector.update(x)
    state, output = new_detector.update(state, np.asarray([x]))

    assert state.grid == [-g - 1 for g in old_detector._state["grid_list"]], (
        "Grid lists do not match between old and new detectors (unknown variance)."
    )

    assert np.allclose(state.current_score_state.stats, old_detector._state["sum"]), (
        "Running sufficient statistics do not match (unknown variance)."
    )

    for old_stats, new_stats in zip(
        old_detector._state["sum_pre_list"], state.previous_score_states
    ):
        assert np.allclose(old_stats, new_stats.stats), (
            "Prefix sufficient statistics do not match (unknown variance)."
        )

# Add one extreme final observation and compare max score statistics directly.
x_extreme = 1_000_000.0
old_alarm = old_detector.update(x_extreme)
state, output = new_detector.update(state, np.asarray([x_extreme]))

assert old_alarm and output["alarm"], (
    "Both detectors should alarm on extreme observation."
)
assert np.isclose(old_detector.max_statistic, output["max_score"]), (
    f"Max score mismatch: old={old_detector.max_statistic}, new={output['max_score']}"
)

print("Unknown-variance old/new API checks passed, including extreme-point max score.")

Unknown-variance old/new API checks passed, including extreme-point max score.


## New API calibration + run (old sandbox equivalent)
This section mirrors the old API sandbox flow for univariate mean-change detection, but uses the new API and `gridcp.calibration` helpers.

In [1]:
stream_len = 500
n_paths_calibrate = 1000
n_paths_eval = 200
changepoint = stream_len // 2

In [2]:
import numpy as np

from gridcp.detector import GridDetector
from gridcp.scores import CUSUM, GaussianMean
from gridcp.calibration import (
    calibrate_threshold_false_alarm,
    draw_samples,
    with_calibrated_threshold,
)


def null_sampler(rng):
    return rng.normal(loc=0.0, scale=1.0)


def pre_change_sampler(rng):
    return rng.normal(loc=0.0, scale=1.0)


def post_change_sampler(rng):
    return rng.normal(loc=2.0, scale=1.0)


rng_cal = np.random.default_rng(42)
rng_data = np.random.default_rng(123)

In [ ]:
# Known variance: calibrate threshold under null.
known_score = CUSUM(n_features=1)
critical_value_known_var = calibrate_threshold_false_alarm(
    known_score,
    false_alarm_probability=0.05,
    stream_len=stream_len,
    n_paths=n_paths_calibrate,
    pre_sampler=null_sampler,
    rng=rng_cal,
    n_features=1,
)
known_detector = with_calibrated_threshold(
    GridDetector(score=known_score, threshold=1.0),
    critical_value_known_var,
)
print("Known-variance critical value:", critical_value_known_var)

# Simulate changed data and estimate average detection delay.
paths = draw_samples(
    n_paths=n_paths_eval,
    stream_len=stream_len,
    n_features=1,
    pre_sampler=pre_change_sampler,
    post_sampler=post_change_sampler,
    changepoint=changepoint,
    rng=rng_data,
)

detect_times = np.ones(n_paths_eval, dtype=np.int64) * stream_len
for path_idx in range(n_paths_eval):
    state = known_detector.init_state()
    for t in range(stream_len):
        state, out = known_detector.update(state, paths[path_idx, t])
        if out["alarm"]:
            detect_times[path_idx] = t + 1
            break

print("Known-variance avg detection delay:", np.mean(detect_times) - changepoint)

Known-variance critical value: 3.0597858330282586
Known-variance avg detection delay: -9.995000000000005


In [9]:
detect_times

array([257, 255, 255, 258, 260, 261, 252, 253, 255, 255, 255, 252, 254,
       253, 261, 257, 258, 255, 259, 257, 256, 262, 262, 253, 255, 253,
       253, 256, 254, 260, 255, 253, 254,   2, 258, 258, 253, 254, 261,
         3, 252, 252, 256,   2, 254, 259, 259, 255, 254, 258, 262, 257,
       256,   2, 256,   6, 252, 253, 252, 257,  38, 253, 260, 251, 259,
       256, 254, 259, 258, 258, 258, 254, 260, 260, 258, 254, 257,   2,
       256, 256, 261, 256, 256, 255, 259, 258, 263, 251, 257, 255, 257,
       258, 254, 257, 254, 253, 254, 252, 254, 262, 259, 254, 258, 258,
       257, 251, 257, 257, 252, 259, 259, 254, 260, 253, 258, 255, 256,
       263, 254, 256, 260, 256, 252, 255, 254, 254, 252, 253, 256, 259,
       256, 255, 263, 256, 260, 255, 254, 258, 254, 255, 253, 259, 255,
        12, 253, 254, 258, 254, 256,   8, 257,   2, 257, 258, 256, 254,
       257,   3, 257, 258, 257, 257, 258, 255, 260, 261, 254, 256, 257,
         9, 259, 261, 253, 256, 259, 252, 253, 252, 260, 256, 25

In [ ]:
import gridcp.old_api as old_api

detector = old_api.make_univariate_mean_change_detector()
start = time.perf_counter()
detector.calibrate_false_alarm(
    false_alarm_probability=0.05,
    N=stream_len,
    K=n_paths_calibrate,
    null_dist=np.random.normal,
)
stop = time.perf_counter()
print(detector._state["penalty_constant"])

2.939190545771412


In [ ]:
# Unknown variance: calibrate threshold under null.
unknown_score = GaussianMean(n_features=1)
critical_value_unknown_var = calibrate_threshold_false_alarm(
    unknown_score,
    false_alarm_probability=0.05,
    stream_len=stream_len,
    n_paths=n_paths_calibrate,
    pre_sampler=null_sampler,
    rng=np.random.default_rng(84),
    n_features=1,
)
unknown_detector = with_calibrated_threshold(
    GridDetector(score=unknown_score, threshold=1.0),
    critical_value_unknown_var,
)
print("Unknown-variance critical value:", critical_value_unknown_var)

# Reuse the same changed sample paths to estimate detection delay.
detect_times_unknown = np.ones(n_paths_eval, dtype=np.int64) * stream_len
for path_idx in range(n_paths_eval):
    state = unknown_detector.init_state()
    for t in range(stream_len):
        state, out = unknown_detector.update(state, paths[path_idx, t])
        if out["alarm"]:
            detect_times_unknown[path_idx] = t + 1
            break

print(
    "Unknown-variance avg detection delay:",
    np.mean(detect_times_unknown) - changepoint,
)

Unknown-variance critical value: 2.5327244321434366
Unknown-variance avg detection delay: -5.219999999999999
